# 06 — Saving and Exporting

Once you have a CP you like, you'll want to send it somewhere — a plotter, a folder simulator, or a 3D printer. This notebook covers the export formats `eucare` ships with:

- `.heg` — eucare's native YAML serialization (full round-trip).
- SVG — vector for laser cutters / pen plotters.
- Matplotlib snapshots.

STL via marching cubes and the FOLD format are listed under the `[threed]` install extra and `docs/improvements.md` respectively.

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    plotting,
    reciprocal_figures,
    rendering,
)


def plot_g(G, ax=None, color='black', linewidth=1.0):
    """Draw the edges of a half-edge graph G on `ax` (or the current axes)."""
    if ax is None:
        ax = plt.gca()
    lines = np.array([
        [G.geometry.to_euclidean(h.orig['pos']),
         G.geometry.to_euclidean(h.dest['pos'])]
        for h in G.halfedges_representing_edges()
    ])
    plotting.plot_lines(lines, ax=ax, colors=color, linewidths=linewidth)
    plotting.set_equal_aspect(ax)
    ax.axis('off')


In [ ]:
from eucare.search_trees import face_bfs_tree
from eucare.reciprocal_figures import assign_this_way_by_face_z_order, make_SRG


def srg_pipeline(G):
    """Run the standard SRG pipeline: BFS z-order -> SRG -> recompute."""
    central = min(G.faces, key=lambda f: np.linalg.norm(f.midpoint()))
    central['z_order'] = 0
    for orig, dest in face_bfs_tree(central):
        dest['z_order'] = orig['z_order'] + 1
    assign_this_way_by_face_z_order(G)
    SRG = make_SRG(G)
    SRG.recompute_lengths_and_angles()
    return SRG


In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G.recompute_lengths_and_angles()
SRG = srg_pipeline(G)
print('SRG:', len(SRG.vertices), 'vertices,', len(SRG.faces), 'faces')


## .heg save

Eucare's native YAML serialization captures the full graph structure plus arbitrary attributes.

Note: the load path currently uses `yaml.SafeLoader`, so graphs with non-trivial Python attributes (tuples, numpy scalars) save but don't round-trip yet — see `docs/improvements.md` (P2.5).

In [ ]:
import tempfile, os
from eucare import io

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'pattern.heg')
    io.save_graph(path, SRG)
    size = os.path.getsize(path)
print(f'wrote {size} bytes')


## SVG export

The simplest route is matplotlib's SVG backend — no extra deps. For laser-cutter / pen-plotter pipelines, see `eucare.rendering.SvgwriteRenderer`, which splits the output into interior / border / drawing-edge files.

In [ ]:
import tempfile, os

fig, ax = plt.subplots(figsize=(5, 5))
plot_g(SRG, ax=ax)
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'pattern.svg')
    fig.savefig(path)
    head = open(path).read()[:200]
plt.close(fig)
print(head)


## A quick matplotlib summary plot

When you don't need vector output, the `plot_g` helper from the setup cell is a one-liner.

In [ ]:
plot_g(SRG); plt.title('final CP'); plt.show()


## Other formats and tools

- **FOLD format** (`fold format.ipynb` in legacy notebooks) — the community standard for flat-foldable patterns. Not yet first-class in `eucare.io`; tracked in [`docs/improvements.md`](../improvements.md).
- **3D / STL** — `eucare.marching_cubes` + the optional `[threed]` install extra (`uv pip install -e '.[threed]'`).
- **Plotter hardware** — see `how_to_connect_plotter.txt` in the repo root for HP-GL setup tips.

## Wrap-up

That completes the curated tour. From here:

- pick a tiling (notebooks 01, 02),
- pick or compose Conway operators (notebook 05),
- run SRG (03),
- fold and check (04),
- export (this one).
